## Análise de dados de locação de bicicletas

### 1. Problema de negócio

*  Caracterizar o perfil de locação de bicicletas no 3º trimestre de 2023.

**Tópicos para análise**
* 5.1- quantiade de locações 
* 5.2- quantidade de devoluções 
* 5.3- %  de clientes 
* 5.4- %  de tipo de bicicleta
* 5.5- %  de tipo dias da semana
* 5.6- %  de horários de início
* 5.7- identificar o perfil de locação por tipo de bicicleta e cliente(top3)
* 5.8- identificar o perfil de locação por estação (top5)
* 5.9 - identificar o perfil de devoluções por estação (top5)

### 2. Import bibliotecas

In [1]:
# imports
import pandas as pd
import numpy as np
from datetime import datetime
import locale
locale.setlocale(locale.LC_ALL, 'en_US.utf8') # setar locale para inglês dos Estados Unidos, para buscar os dias da semana corretamente


'en_US.utf8'

### 3. Carga de dados

In [ ]:
# import dados

df_jul = pd.read_csv('202307-divvy-tripdata.csv')
df_ago = pd.read_csv('202308-divvy-tripdata.csv')
df_set = pd.read_csv('202309-divvy-tripdata.csv')


### 4. Análise exploratória de dados

In [ ]:
# junta os arquivos em um único arquivo
df = pd.concat(([df_jul, df_ago, df_set]))


# lista as 3 primeiras linhas do df
df.head(3)

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,9340B064F0AEE130,electric_bike,2023-07-23 20:06:14,2023-07-23 20:22:44,Kedzie Ave & 110th St,20204,Public Rack - Racine Ave & 109th Pl,877,41.692406,-87.700905,41.694835,-87.653041,member
1,D1460EE3CE0D8AF8,classic_bike,2023-07-23 17:05:07,2023-07-23 17:18:37,Western Ave & Walton St,KA1504000103,Milwaukee Ave & Grand Ave,13033,41.898418,-87.686596,41.891578,-87.648384,member
2,DF41BE31B895A25E,classic_bike,2023-07-23 10:14:53,2023-07-23 10:24:29,Western Ave & Walton St,KA1504000103,Damen Ave & Pierce Ave,TA1305000041,41.898418,-87.686596,41.909396,-87.677692,member


In [4]:
# checa quantidade de linhas e colunas do df
df.shape

(2205714, 13)

In [5]:
# verifica se há dados duplicados
df[df.duplicated()]

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual


In [6]:
# conta valores únicos  por coluna e ordena em ordem decrescente
df.nunique().sort_values(ascending=False)

ride_id               2205714
ended_at              1793010
started_at            1788996
start_lat              450440
start_lng              432518
end_lat                  1826
end_lng                  1822
start_station_name       1435
end_station_name         1434
end_station_id           1386
start_station_id         1382
rideable_type               3
member_casual               2
dtype: int64

In [7]:
# checa quantidade de valores ausentes
df.isna().sum().sort_values(ascending=False)

end_station_name      363163
end_station_id        363163
start_station_name    343174
start_station_id      343174
end_lat                 3349
end_lng                 3349
ride_id                    0
rideable_type              0
started_at                 0
ended_at                   0
start_lat                  0
start_lng                  0
member_casual              0
dtype: int64

In [8]:
# checa tipos de dados por coluna
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2205714 entries, 0 to 666370
Data columns (total 13 columns):
 #   Column              Dtype  
---  ------              -----  
 0   ride_id             object 
 1   rideable_type       object 
 2   started_at          object 
 3   ended_at            object 
 4   start_station_name  object 
 5   start_station_id    object 
 6   end_station_name    object 
 7   end_station_id      object 
 8   start_lat           float64
 9   start_lng           float64
 10  end_lat             float64
 11  end_lng             float64
 12  member_casual       object 
dtypes: float64(4), object(9)
memory usage: 235.6+ MB


### 5. Limpeza e tratamento de dados


In [9]:
# converter data de objeto para datetime
df['started_at'] = pd.to_datetime(df['started_at'])
df['ended_at'] = pd.to_datetime(df['ended_at'])


In [10]:
# extrair datas de inicio e conclusao

df['data_inicio']= df['started_at'].dt.strftime("%d/%m/%Y")
df['data_conclusao']= df['ended_at'].dt.strftime("%d/%m/%Y")

df.head(2)

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,data_inicio,data_conclusao
0,9340B064F0AEE130,electric_bike,2023-07-23 20:06:14,2023-07-23 20:22:44,Kedzie Ave & 110th St,20204,Public Rack - Racine Ave & 109th Pl,877,41.692406,-87.700905,41.694835,-87.653041,member,23/07/2023,23/07/2023
1,D1460EE3CE0D8AF8,classic_bike,2023-07-23 17:05:07,2023-07-23 17:18:37,Western Ave & Walton St,KA1504000103,Milwaukee Ave & Grand Ave,13033,41.898418,-87.686596,41.891578,-87.648384,member,23/07/2023,23/07/2023


In [11]:

# extrair nome do dia da semana
df['dia_inicio'] = df['started_at'].dt.day_name()
df['dia_conclusao'] = df['ended_at'].dt.day_name()
df['mes_inicio'] = df['started_at'].dt.month
df['mes_fim'] = df['ended_at'].dt.month

df.head(2)


,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,data_inicio,data_conclusao,dia_inicio,dia_conclusao,mes_inicio,mes_fim
0,9340B064F0AEE130,electric_bike,2023-07-23 20:06:14,2023-07-23 20:22:44,Kedzie Ave & 110th St,20204,Public Rack - Racine Ave & 109th Pl,877,41.692406,-87.700905,41.694835,-87.653041,member,23/07/2023,23/07/2023,Sunday,Sunday,7,7
1,D1460EE3CE0D8AF8,classic_bike,2023-07-23 17:05:07,2023-07-23 17:18:37,Western Ave & Walton St,KA1504000103,Milwaukee Ave & Grand Ave,13033,41.898418,-87.686596,41.891578,-87.648384,member,23/07/2023,23/07/2023,Sunday,Sunday,7,7


In [12]:
# extrair horário
df['horario_inicio']= df['started_at'].dt.time
df['horario_fim'] = df['ended_at'].dt.time
df['hora_inicio']= df['started_at'].dt.hour
df['hora_fim']= df['ended_at'].dt.hour

df.head(2)

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,...,data_inicio,data_conclusao,dia_inicio,dia_conclusao,mes_inicio,mes_fim,horario_inicio,horario_fim,hora_inicio,hora_fim
0,9340B064F0AEE130,electric_bike,2023-07-23 20:06:14,2023-07-23 20:22:44,Kedzie Ave & 110th St,20204,Public Rack - Racine Ave & 109th Pl,877,41.692406,-87.700905,...,23/07/2023,23/07/2023,Sunday,Sunday,7,7,20:06:14,20:22:44,20,20
1,D1460EE3CE0D8AF8,classic_bike,2023-07-23 17:05:07,2023-07-23 17:18:37,Western Ave & Walton St,KA1504000103,Milwaukee Ave & Grand Ave,13033,41.898418,-87.686596,...,23/07/2023,23/07/2023,Sunday,Sunday,7,7,17:05:07,17:18:37,17,17


In [13]:
# preenche a coluna com dados vazios para estação de início e de fim

df = df.fillna({'end_station_name': 'N/A', 'start_station_name': 'N/A'})


In [14]:
# renomeia os nomes das colunas
df.rename(columns={ 'ride_id': 'id_locacao',
                    'rideable_type': 'tipo_bicicleta',
                    'started_at': 'iniciada_em',
                    'ended_at': 'concluida_em',
                    'start_station_id': 'id_estacao_inicio',
                    'start_station_name': 'nome_estacao_inicio',
                    'started_at': 'iniciada_em',
                    'end_station_name': 'nome_estacao_fim',
                    'end_station_id': 'id_estacao_fim',
                    'member_casual': 'tipo_cliente'
                   },
           inplace = True)      
df.head(2)

,id_locacao,tipo_bicicleta,iniciada_em,concluida_em,nome_estacao_inicio,id_estacao_inicio,nome_estacao_fim,id_estacao_fim,start_lat,start_lng,...,data_inicio,data_conclusao,dia_inicio,dia_conclusao,mes_inicio,mes_fim,horario_inicio,horario_fim,hora_inicio,hora_fim
0,9340B064F0AEE130,electric_bike,2023-07-23 20:06:14,2023-07-23 20:22:44,Kedzie Ave & 110th St,20204,Public Rack - Racine Ave & 109th Pl,877,41.692406,-87.700905,...,23/07/2023,23/07/2023,Sunday,Sunday,7,7,20:06:14,20:22:44,20,20
1,D1460EE3CE0D8AF8,classic_bike,2023-07-23 17:05:07,2023-07-23 17:18:37,Western Ave & Walton St,KA1504000103,Milwaukee Ave & Grand Ave,13033,41.898418,-87.686596,...,23/07/2023,23/07/2023,Sunday,Sunday,7,7,17:05:07,17:18:37,17,17


In [15]:
# renomeia dados de inglês par português
df.replace({'Sunday': 'Domingo', # dias da semana
            'Monday': 'Segunda',
            'Tuesday': 'Terça',
            'Wednesday': 'Quarta',
            'Friday': 'Sexta',
            'Thursday': 'Quinta',
            'Saturday': 'Sábado',
            'member':'membro', # tipo_cliente            
            'electric_bike': 'elétrica', # tipos de bicicletas
            'classic_bike': 'clássica',
            'docked_bike': 'docked',
            
},
            inplace=True
)
df.head(2)

,id_locacao,tipo_bicicleta,iniciada_em,concluida_em,nome_estacao_inicio,id_estacao_inicio,nome_estacao_fim,id_estacao_fim,start_lat,start_lng,...,data_inicio,data_conclusao,dia_inicio,dia_conclusao,mes_inicio,mes_fim,horario_inicio,horario_fim,hora_inicio,hora_fim
0,9340B064F0AEE130,elétrica,2023-07-23 20:06:14,2023-07-23 20:22:44,Kedzie Ave & 110th St,20204,Public Rack - Racine Ave & 109th Pl,877,41.692406,-87.700905,...,23/07/2023,23/07/2023,Domingo,Domingo,7,7,20:06:14,20:22:44,20,20
1,D1460EE3CE0D8AF8,clássica,2023-07-23 17:05:07,2023-07-23 17:18:37,Western Ave & Walton St,KA1504000103,Milwaukee Ave & Grand Ave,13033,41.898418,-87.686596,...,23/07/2023,23/07/2023,Domingo,Domingo,7,7,17:05:07,17:18:37,17,17


In [16]:
# remover colunas desnecessárias

df = df.drop(columns=['end_lat','end_lng', 'start_lat','start_lng', 'iniciada_em', 'concluida_em', 'id_estacao_inicio', 'id_estacao_fim' ], axis = 1)
df.head(3)

,id_locacao,tipo_bicicleta,nome_estacao_inicio,nome_estacao_fim,tipo_cliente,data_inicio,data_conclusao,dia_inicio,dia_conclusao,mes_inicio,mes_fim,horario_inicio,horario_fim,hora_inicio,hora_fim
0,9340B064F0AEE130,elétrica,Kedzie Ave & 110th St,Public Rack - Racine Ave & 109th Pl,membro,23/07/2023,23/07/2023,Domingo,Domingo,7,7,20:06:14,20:22:44,20,20
1,D1460EE3CE0D8AF8,clássica,Western Ave & Walton St,Milwaukee Ave & Grand Ave,membro,23/07/2023,23/07/2023,Domingo,Domingo,7,7,17:05:07,17:18:37,17,17
2,DF41BE31B895A25E,clássica,Western Ave & Walton St,Damen Ave & Pierce Ave,membro,23/07/2023,23/07/2023,Domingo,Domingo,7,7,10:14:53,10:24:29,10,10


In [17]:
# mostra breve resumo dos dados
df.describe(exclude='number')

,id_locacao,tipo_bicicleta,nome_estacao_inicio,nome_estacao_fim,tipo_cliente,data_inicio,data_conclusao,dia_inicio,dia_conclusao,horario_inicio,horario_fim
count,2205714,2205714,2205714,2205714,2205714,2205714,2205714,2205714,2205714,2205714,2205714
unique,2205714,3,1436,1435,2,92,99,7,7,84823,84935
top,9340B064F0AEE130,clássica,N/A,N/A,membro,12/08/2023,12/08/2023,Sábado,Sábado,17:22:57,17:29:13
freq,1,1106463,343174,363163,1301591,33536,33410,384231,383333,90,92


### 5. Análises

*  Caracterizar o perfil de locação de bicicletas no 3º trimestre de 2023.

**Tópicos para análise**
* 5.1- quantiade de locações 
* 5.2- quantidade de devoluções 
* 5.3- %  de clientes 
* 5.4- %  de tipo de bicicleta
* 5.5- %  de tipo dias da semana
* 5.6- %  de horários de início
* 5.7- identificar o perfil de locação por tipo de bicicleta e cliente(top3)
* 5.8- identificar o perfil de locação por estação (top5)
* 5.9 - identificar o perfil de devoluções por estação (top5)

In [69]:
# 5.1 Total de locações no trimestre 
locacoes_total=df['id_locacao'].count()
locacoes_total

2205714

In [ ]:
# 5.2 Total de devoluções no trimestre 
df_estacao_fim=df.loc[df['nome_estacao_fim'] != "N/A"]
devolucoes = df_estacao_fim['nome_estacao_fim'].count()
devolucoes


1842551

In [73]:
#5.2 % de devoluções

percentual_devolucoes = round((devolucoes/locacoes_total)*100,2)
percentual_devolucoes

83.54

In [63]:
# 5.3- %  de clientes

df_cliente= round((df['tipo_cliente'].value_counts()/df.shape[0]*100),2).to_frame('%').reset_index()
df_cliente

,tipo_cliente,%
0,membro,59.01
1,casual,40.99


In [ ]:
# 5.4 - % de bicicletas locadas 
percentual_locacoes_mes= round((df[['mes_inicio']].value_counts()/df.shape[0]*100),2).to_frame('% locação').sort_values(by='% locação',ascending=False).reset_index()
percentual_locacoes_mes

,mes_inicio,% locação
0,8,34.99
1,7,34.80
2,9,30.21


In [ ]:
# 5.4 - % de bicicletas 
percentual_devolvidas_mes= round((df[['mes_fim']].value_counts()/df.shape[0]*100),2).to_frame('% devoluções').sort_values(by='% devoluções',ascending=False).reset_index()
percentual_devolvidas_mes

,mes_fim,% devoluções
0,8,34.99
1,7,34.80
2,9,30.21
3,10,0.01


In [22]:
# 5.4- %  de tipo de bicicleta

percentual_tipo_bike= round((df['tipo_bicicleta'].value_counts()/df.shape[0]*100),2).to_frame('%').reset_index()
percentual_tipo_bike

,tipo_bicicleta,%
0,clássica,50.16
1,elétrica,48.28
2,docked,1.56


In [23]:
# quantidade de bicicletas por tipo
qtde_tipo_bike = df['tipo_bicicleta'].value_counts().to_frame('qtde de locações').reset_index()
qtde_tipo_bike

,tipo_bicicleta,qtde de locações
0,clássica,1106463
1,elétrica,1064870
2,docked,34381


In [24]:
# 5.5- %  de locação por dia da semana

percentual_dias_locacao = round(df.groupby('dia_inicio')['id_locacao'].count()/df.shape[0]*100,2).to_frame('% de locações').sort_values(by='% de locações',ascending=False).reset_index()
percentual_dias_locacao 


,dia_inicio,% de locações
0,Sábado,17.42
1,Sexta,15.12
2,Quinta,14.35
3,Terça,13.72
4,Domingo,13.67
5,Quarta,13.48
6,Segunda,12.24


In [25]:
# 5.5- %  de devolução por dia da semana

percentual_dias_devolução = round(df.groupby('dia_conclusao')['id_locacao'].count()/df.shape[0]*100,2).to_frame('% de devoluções').sort_values(by='% de devoluções',ascending=False).reset_index()
percentual_dias_devolução 

,dia_conclusao,% de devoluções
0,Sábado,17.38
1,Sexta,15.07
2,Quinta,14.34
3,Domingo,13.75
4,Terça,13.72
5,Quarta,13.48
6,Segunda,12.27


In [26]:
# 5.6- %  de horários de início (top5)
hora_inicio_locacao = round(df.groupby('hora_inicio')['id_locacao'].count()/df.shape[0]*100,2).to_frame('% de locações').sort_values(by='% de locações',ascending=False).reset_index().head(5)
hora_inicio_locacao

,hora_inicio,% de locações
0,17,10.07
1,18,8.79
2,16,8.56
3,15,6.85
4,19,6.54


In [27]:
# checa top 5 horários por quantidade de locações por horários de pico
horarios= df.groupby(['hora_inicio'])['id_locacao'].count().sort_values(ascending=False).to_frame('Qtde_locações').head(5)
horarios

,Qtde_locações
hora_inicio,
17,222128
18,193836
16,188699
15,151059
19,144213


In [28]:
# checa top 5 horários por quantidade de devoluções por horários de pico
horarios_devolucoes= df.groupby(['hora_fim'])['id_locacao'].count().sort_values(ascending=False).to_frame('Qtde_devoluções').head(5)
horarios_devolucoes

,Qtde_devoluções
hora_fim,
17,220067
18,202532
16,180033
19,156896
15,145783


In [29]:
# 5.7- identificar o perfil de locação por tipo de bicicleta e cliente(top3)
condicao1 = df['tipo_bicicleta'] == 'clássica'
condicao2 = df['tipo_cliente'] == 'membro'
df_classica = df.loc[(condicao1)& (condicao2)]
df_classica_membro= round((df_classica[['tipo_bicicleta','tipo_cliente','dia_inicio','hora_inicio']].value_counts()/df.shape[0]*100),2).to_frame('%').reset_index().head(3)
df_classica_membro

,tipo_bicicleta,tipo_cliente,dia_inicio,hora_inicio,%
0,clássica,membro,Terça,17,0.61
1,clássica,membro,Quarta,17,0.61
2,clássica,membro,Quinta,17,0.59


In [30]:
# 5.7- identificar o perfil de locação por tipo de bicicleta e cliente(top3)
condicao1 = df['tipo_bicicleta'] == 'clássica'
condicao2 = df['tipo_cliente'] == 'casual'
df_classica = df.loc[(condicao1)& (condicao2)]
df_classica_casual= round((df_classica[['tipo_bicicleta','tipo_cliente','dia_inicio','hora_inicio']].value_counts()/df.shape[0]*100),2).to_frame('%').reset_index().head(3)
df_classica_casual

,tipo_bicicleta,tipo_cliente,dia_inicio,hora_inicio,%
0,clássica,casual,Sábado,15,0.38
1,clássica,casual,Sábado,14,0.38
2,clássica,casual,Sábado,13,0.37


In [31]:
# 5.7- identificar o perfil de locação por tipo de bicicleta e cliente(top3)
condicao1 = df['tipo_bicicleta'] == 'elétrica'
condicao2 = df['tipo_cliente'] == 'membro'
df_eletrica = df.loc[(condicao1)& (condicao2)]
df_eletrica_membro= round((df_eletrica[['tipo_bicicleta','tipo_cliente','dia_inicio','hora_inicio']].value_counts()/df.shape[0]*100),2).to_frame('%').reset_index().head(3)
df_eletrica_membro

,tipo_bicicleta,tipo_cliente,dia_inicio,hora_inicio,%
0,elétrica,membro,Quarta,17,0.51
1,elétrica,membro,Quinta,17,0.50
2,elétrica,membro,Terça,17,0.49


In [32]:
# 5.7- identificar o perfil de locação por tipo de bicicleta e cliente(top3)
condicao1 = df['tipo_bicicleta'] == 'elétrica'
condicao2 = df['tipo_cliente'] == 'casual'
df_eletrica = df.loc[(condicao1)& (condicao2)]
df_eletrica_casual= round((df_eletrica[['tipo_bicicleta','tipo_cliente','dia_inicio','hora_inicio']].value_counts()/df.shape[0]*100),2).to_frame('%').reset_index().head(3)
df_eletrica_casual

,tipo_bicicleta,tipo_cliente,dia_inicio,hora_inicio,%
0,elétrica,casual,Sexta,17,0.31
1,elétrica,casual,Sábado,15,0.31
2,elétrica,casual,Sábado,14,0.30


In [33]:
# 5.7- identificar o perfil de locação por tipo de bicicleta e cliente(top3)
condicao1 = df['tipo_bicicleta'] == 'docked'
condicao2 = df['tipo_cliente'] == 'membro'
df_docked = df.loc[(condicao1)& (condicao2)]
df_docked_membro= round((df_docked[['tipo_bicicleta','tipo_cliente','dia_inicio','hora_inicio']].value_counts()/df.shape[0]*100),2).to_frame('%').reset_index().head(3)
df_docked_membro

,tipo_bicicleta,tipo_cliente,dia_inicio,hora_inicio,%


In [34]:
# 5.7- identificar o perfil de locação por tipo de bicicleta e cliente(top3)
condicao1 = df['tipo_bicicleta'] == 'docked'
condicao2 = df['tipo_cliente'] == 'casual'
df_docked = df.loc[(condicao1)& (condicao2)]
df_docked_casual= round((df_docked[['tipo_bicicleta','tipo_cliente','dia_inicio','hora_inicio']].value_counts()/df.shape[0]*100),2).to_frame('%').reset_index().head(3)
df_docked_casual

,tipo_bicicleta,tipo_cliente,dia_inicio,hora_inicio,%
0,docked,casual,Sábado,15,0.03
1,docked,casual,Sábado,13,0.03
2,docked,casual,Sábado,14,0.03


In [35]:
# 5.8- identificar o perfil de locação por estacão (top5)

percentual_locacao_estacao_inicio = round(df.groupby('nome_estacao_inicio')['id_locacao'].count()/df.shape[0]*100,2).to_frame('% locações iniciadas').sort_values(by='% locações iniciadas',ascending=False).reset_index().head(5)
percentual_locacao_estacao_inicio


,nome_estacao_inicio,% locações iniciadas
0,N/A,15.56
1,Streeter Dr & Grand Ave,1.41
2,DuSable Lake Shore Dr & Monroe St,0.86
3,DuSable Lake Shore Dr & North Blvd,0.85
4,Michigan Ave & Oak St,0.83


In [75]:
# 5.8- identificar o perfil de locação por estacão (top5)

estacao_inicio = df.groupby(['nome_estacao_inicio','tipo_cliente','tipo_bicicleta'])['id_locacao'].count().sort_values(ascending=False).reset_index().head(5)
estacao_inicio

,nome_estacao_inicio,tipo_cliente,tipo_bicicleta,id_locacao
0,N/A,membro,elétrica,200665
1,N/A,casual,elétrica,142509
2,Streeter Dr & Grand Ave,casual,clássica,15564
3,DuSable Lake Shore Dr & Monroe St,casual,clássica,9357
4,Michigan Ave & Oak St,casual,clássica,7604


In [37]:
# 5.8- identificar o perfil de locação por estacão (top5)

estacao_inicio = df.groupby(['nome_estacao_inicio','hora_inicio'])['id_locacao'].count().sort_values(ascending=False).reset_index().head(5)
estacao_inicio

,nome_estacao_inicio,hora_inicio,id_locacao
0,N/A,17,30491
1,N/A,18,28492
2,N/A,16,28099
3,N/A,15,23866
4,N/A,19,23579


In [38]:
# 5.8- identificar o perfil de locação por estacão (top5)

estacao_inicio = df.groupby(['nome_estacao_inicio','dia_inicio'])['id_locacao'].count().sort_values(ascending=False).reset_index().head(5)
estacao_inicio

,nome_estacao_inicio,dia_inicio,id_locacao
0,N/A,Sábado,62151
1,N/A,Sexta,54804
2,N/A,Quinta,48613
3,N/A,Domingo,47443
4,N/A,Terça,45392


In [39]:
# 5.9- identificar o perfil de devolução por estacão (top5)

percentual_locacao_estacao_fim = round(df.groupby('nome_estacao_fim')['id_locacao'].count()/df.shape[0]*100,2).to_frame('% locações finalizadas').sort_values(by='% locações finalizadas',ascending=False).reset_index().head(5)
percentual_locacao_estacao_fim


,nome_estacao_fim,% locações finalizadas
0,N/A,16.46
1,Streeter Dr & Grand Ave,1.42
2,DuSable Lake Shore Dr & North Blvd,0.95
3,Michigan Ave & Oak St,0.84
4,DuSable Lake Shore Dr & Monroe St,0.82


In [40]:
# 5.9- identificar o perfil de devolução por estacão (top5)

estacao_fim = df.groupby(['nome_estacao_fim','tipo_cliente','tipo_bicicleta'])['id_locacao'].count().sort_values(ascending=False).reset_index().head(5)
estacao_fim

,nome_estacao_fim,tipo_cliente,tipo_bicicleta,id_locacao
0,N/A,membro,elétrica,197763
1,N/A,casual,elétrica,162135
2,Streeter Dr & Grand Ave,casual,clássica,16604
3,DuSable Lake Shore Dr & North Blvd,casual,clássica,9088
4,DuSable Lake Shore Dr & Monroe St,casual,clássica,8610


In [41]:
# 5.9- identificar o perfil de devolução por estacão (top5)

estacao_inicio = df.groupby(['nome_estacao_fim','hora_fim'])['id_locacao'].count().sort_values(ascending=False).reset_index().head(5)
estacao_inicio

,nome_estacao_fim,hora_fim,id_locacao
0,N/A,18,34626
1,N/A,17,33276
2,N/A,16,27188
3,N/A,19,26987
4,N/A,15,23241


In [42]:
# 5.9- identificar o perfil de devolução por estacão (top5)

estacao_fim = df.groupby(['nome_estacao_fim','dia_conclusao'])['id_locacao'].count().sort_values(ascending=False).reset_index().head(5)
estacao_fim

,nome_estacao_fim,dia_conclusao,id_locacao
0,N/A,Sábado,65387
1,N/A,Sexta,58189
2,N/A,Quinta,51605
3,N/A,Domingo,49926
4,N/A,Terça,48166
